# Recuperação de Informação: TF-IDF e BM25

Este notebook implementa e compara dois modelos clássicos de recuperação de informação baseados em ponderação de termos:

- **TF-IDF** (Term Frequency–Inverse Document Frequency) — modelo vetorial clássico
- **BM25** (Okapi BM25) — extensão probabilística com saturação de TF e normalização de comprimento

Ambos representam documentos como vetores e usam **similaridade de cosseno** para ranquear candidatos.

> Referência: *Speech and Language Processing*, Jurafsky & Martin, 3ª ed., Cap. 14.

In [1]:
from corpus import query_tfidf_v1, dataset_tfidf_v1, dataset_IR_v1

## 1. Corpus

Usamos dois corpora distintos ao longo do notebook:

| Variável | Idioma | Uso |
|---|---|---|
| `dataset_tfidf_v1` | Inglês | Inspecionar os scores numericamente (corpus pequeno e controlado) |
| `dataset_IR_v1` | Português | Testar a recuperação em linguagem natural |

In [2]:
print(dataset_tfidf_v1)

['Sweet sweet nurse!Love?', 'Sweet sorrow', 'How sweet is love?', 'Nurse!']


In [3]:
print(query_tfidf_v1)

['sweet love']


## 2. TF-IDF

O modelo **vetorial** representa cada documento como um vetor de pesos no espaço de termos do vocabulário. O peso de cada termo $t$ no documento $d$ é o produto:

$$\text{TF-IDF}(t, d) = \text{TF}(t, d) \times \text{IDF}(t)$$

**TF — Term Frequency** (normalização logarítmica):

$$\text{TF}(t, d) = \begin{cases} 1 + \log_{10}(\text{count}(t, d)) & \text{se } \text{count}(t,d) > 0 \\ 0 & \text{caso contrário} \end{cases}$$

**IDF — Inverse Document Frequency**:

$$\text{IDF}(t) = \log_{10}\!\left(\frac{N}{df_t}\right)$$

onde $N$ é o número total de documentos e $df_t$ é o número de documentos que contêm o termo $t$.

Termos raros têm IDF alto (mais discriminativos); termos que aparecem em quase todos os documentos têm IDF próximo de zero.

In [4]:
from tfidf_retriever import TFIDFRetriever
from bm25_retriever import BM25Retriever

In [5]:
retriever = TFIDFRetriever(dataset_tfidf_v1)

### 2.1 Pré-processamento e Vocabulário

O `CorpusIndex` (composto internamente pelo `TFIDFRetriever`) realiza os seguintes passos:

1. **Tokenização** via regex `\b\w+\b` — extrai tokens alfanuméricos
2. **Normalização** — converte para minúsculas
3. **Vocabulário** — ordenado alfabeticamente para indexação determinística

In [6]:
retriever.index.preprocessed

[['sweet', 'sweet', 'nurse', 'love'],
 ['sweet', 'sorrow'],
 ['how', 'sweet', 'is', 'love'],
 ['nurse']]

In [7]:
retriever.index.tokens

{'how': 0, 'is': 1, 'love': 2, 'nurse': 3, 'sorrow': 4, 'sweet': 5}

### 2.2 Matriz TF-IDF

A matriz `score_matrix` tem dimensão $M \times N$ (documentos × termos). Cada linha é o vetor TF-IDF de um documento.

Abaixo, o vetor do **documento 0** (`"Sweet sweet nurse!Love?"` → `['sweet', 'sweet', 'nurse', 'love']`).  
Note que `sweet` aparece **2 vezes**, o que aumenta seu TF em relação a `nurse` e `love` (1 vez cada).

In [8]:
retriever.score_matrix[0]

array([0.        , 0.        , 0.30103   , 0.30103   , 0.        ,
       0.16254904])

### 2.3 TF de Termos Específicos

Calculamos o componente TF isolado para verificar os valores esperados:

- `"sweet"` aparece **2 vezes** no doc 0 → $\text{TF} = 1 + \log_{10}(2) \approx 1{,}301$
- `"love"` aparece **1 vez** → $\text{TF} = 1 + \log_{10}(1) = 1{,}0$

In [9]:
retriever._compute_tf(retriever.index.count_matrix[0, retriever.index.tokens["sweet"]])

1.3010299956639813

In [10]:
retriever._compute_tf(retriever.index.count_matrix[0, retriever.index.tokens["love"]])

1.0

In [11]:
retriever.score_matrix[1]

array([0.        , 0.        , 0.        , 0.        , 0.60205999,
       0.12493874])

### 2.4 Recuperação por Similaridade de Cosseno

A consulta é vetorizada com o mesmo esquema TF-IDF e comparada aos documentos candidatos — filtrados previamente pelo **índice invertido** — via similaridade de cosseno:

$$\text{sim}(q, d) = \frac{q \cdot d}{\|q\| \cdot \|d\|}$$

O índice invertido mapeia cada termo aos documentos que o contêm, evitando comparar a consulta com o corpus inteiro.

In [12]:
retriever.retrieve("sweet love")

[('sweet sweet nurse love', np.float64(0.7468654211522299)),
 ('how sweet is love', np.float64(0.3574976313912116))]

## 3. BM25 (Okapi BM25)

O **BM25** é uma função de ranqueamento probabilística que resolve duas limitações do TF-IDF clássico:

1. **Saturação de TF**: um termo que aparece 10 vezes não vale 10× mais do que um que aparece 1 vez — a contribuição cresce de forma sub-linear.
2. **Normalização de comprimento**: documentos mais longos têm naturalmente mais ocorrências; o BM25 penaliza isso.

O score de cada termo $t$ no documento $d$ é:

$$\text{BM25}(t, d) = \text{IDF}(t) \times \frac{f(t,d) \cdot (k_1 + 1)}{f(t,d) + k_1 \cdot \!\left(1 - b + b \cdot \dfrac{|d|}{\text{avgdl}}\right)}$$

onde $f(t,d)$ é a frequência do termo, $|d|$ é o comprimento do documento, $\text{avgdl}$ é o comprimento médio do corpus, e os hiperparâmetros padrão são $k_1 = 1{,}5$ e $b = 0{,}75$.

O IDF do BM25 usa uma formulação mais robusta para termos muito frequentes:

$$\text{IDF}(t) = \log\!\left(\frac{N - df_t + 0{,}5}{df_t + 0{,}5} + 1\right)$$

O vetor de consulta é **binário**: vale 1 para cada termo presente no vocabulário.

In [13]:
bm25 = BM25Retriever(dataset_IR_v1)

### 3.1 Recuperação com BM25

Usamos o `dataset_IR_v1` (corpus em português). Compare os resultados com os que o TF-IDF retornaria para a mesma consulta.

In [14]:
bm25.retrieve("Do que gatos e cachorros gostam ?", top_k=3)

[('gatos e cachorros gostam de comida', np.float64(0.8940200916405727)),
 ('cachorros gostam de ossos', np.float64(0.42124370726515253)),
 ('gatos e cachorros podem ser amigos', np.float64(0.4103797598201705))]